# Vigil — Market Surveillance & Regulatory Reporting Copilot

**Live walkthrough against the real `VIGIL.CORE` schema on Snowflake** — every result below is a
real query result, not a mock. This notebook is Phase 8's demo layer (`plan.md`): a narrated tour
of the schema, RBAC, detectors, and skills built in Phases 1–6, run against the synthetic Japan
dataset loaded in Phase 5.

Nothing here re-implements query logic — every cell either calls a `skills/` module already built
and tested in Phase 6, or queries a `sql/detectors/`/`sql/ddl/` view directly. That's deliberate:
the demo layer is a *consumer* of the schema/detector contract, not a second implementation of it.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import pandas as pd
from run_sql import connect
from skills.surveillance_query import SurveillanceQueryRequest, build_query, companion_coverage_query
from skills.assure_report import assure
from ingest.report_adaptor import TemplateField

pd.set_option("display.max_colwidth", 60)

conn = connect()
cur = conn.cursor()
cur.execute("USE ROLE ACCOUNTADMIN")
cur.execute("USE SECONDARY ROLES NONE")
cur.execute("USE DATABASE VIGIL")
cur.execute("USE SCHEMA CORE")


def q(sql, params=None):
    cur.execute(sql, params or {})
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)


print("Connected.")


Connected.


## 1. Data landscape

Row counts for the Japan synthetic dataset loaded in Phase 5 — 8 venues (6 active, 2 discontinued
with real historical trades preserved before their `DISCONTINUED_AT` date), one regulator (`JP`).


In [2]:
tables = ["JURISDICTIONS", "VENUES", "BENEFICIAL_OWNERS", "INSTRUMENTS", "MARKET_PARTICIPANTS",
          "ORDERS", "TRADES", "POSITIONS", "TRANSACTION_REPORTS", "REPORT_TEMPLATES",
          "DETECTOR_CALIBRATION"]
counts = {t: q(f"SELECT COUNT(*) AS N FROM {t}").iloc[0]["N"] for t in tables}
pd.DataFrame(counts.items(), columns=["TABLE", "ROW_COUNT"])


,TABLE,ROW_COUNT
0,JURISDICTIONS,1
1,VENUES,8
2,BENEFICIAL_OWNERS,10
3,INSTRUMENTS,8
4,MARKET_PARTICIPANTS,37
5,ORDERS,3932
6,TRADES,901
7,POSITIONS,901
8,TRANSACTION_REPORTS,901
9,REPORT_TEMPLATES,4


In [3]:
q("""
SELECT VENUE_ID, VENUE_TYPE, STATUS, ACTIVE_FROM, DISCONTINUED_AT
FROM VENUES_CURRENT ORDER BY STATUS, VENUE_ID
""")


,VENUE_ID,VENUE_TYPE,STATUS,ACTIVE_FROM,DISCONTINUED_AT
0,JPNX,pts,active,None,None
1,ODX,pts,active,None,None
2,ODXST,pts,active,None,None
3,TOCOM,exchange,active,None,None
4,XOSE,exchange,active,None,None
5,XTKS,exchange,active,None,None
6,CBOJ,pts,discontinued,None,2025-08-29
7,CBOJBIDS,block_trading,discontinued,None,2025-08-29


## 2. Trade surveillance — wash trading

`surveillance_query` (the `surveillance-query` skill, Phase 6) structurally pairs every
wash-trading result with its `WASH_DETECTION_COVERAGE` companion (Fix #3) — "no wash trades
found" and "no wash trades could be checked for" are never conflated. This isn't convention
enforced by review; the skill's `companion_coverage_query` makes it the only path available.


In [4]:
req = SurveillanceQueryRequest(detector="wash_trading", jurisdiction_id="JP", flagged_only=False)
wash_df = q(build_query(req))
print(f"{len(wash_df)} wash-trading candidate rows (all candidates, exemptions included)")
print(f"NULL IS_TRIGGER_EXEMPT rows: {wash_df['IS_TRIGGER_EXEMPT'].isna().sum()}  "
      "(fixed this session -- was 6 before the COALESCE fix; see TRACKER.md)")
wash_df[["TRADE_ID_1", "TRADE_ID_2", "VENUE_ID", "CANDIDATE_TYPE", "IS_TRIGGER_EXEMPT"]].head(10)


29 wash-trading candidate rows (all candidates, exemptions included)
NULL IS_TRIGGER_EXEMPT rows: 0  (fixed this session -- was 6 before the COALESCE fix; see TRACKER.md)


,TRADE_ID_1,TRADE_ID_2,VENUE_ID,CANDIDATE_TYPE,IS_TRIGGER_EXEMPT
0,T0000046,T0000046,XOSE,same_row_self_trade,False
1,T0000051,T0000051,XOSE,same_row_self_trade,False
2,T0000110,T0000110,TOCOM,same_row_self_trade,False
3,T0000131,T0000131,XOSE,same_row_self_trade,False
4,T0000138,T0000138,ODX,same_row_self_trade,False
5,T0000213,T0000213,CBOJ,same_row_self_trade,False
6,T0000249,T0000249,XOSE,same_row_self_trade,False
7,T0000266,T0000266,ODXST,same_row_self_trade,False
8,T0000361,T0000361,ODXST,same_row_self_trade,False
9,T0000372,T0000372,CBOJ,same_row_self_trade,False


In [5]:
coverage_df = q(companion_coverage_query(req))
coverage_df.sort_values("PCT_TRADES_WITH_RESOLVABLE_COUNTERPARTY").head(10)


,JURISDICTION_ID,VENUE_ID,TRADE_DATE,TOTAL_TRADES,RESOLVABLE_TRADES,PCT_TRADES_WITH_RESOLVABLE_COUNTERPARTY
97,JP,TOCOM,2026-02-19,1,0,0.000000
454,JP,XTKS,2026-02-25,1,0,0.000000
647,JP,ODX,2026-05-22,1,0,0.000000
646,JP,ODX,2025-12-06,1,0,0.000000
565,JP,ODXST,2025-07-02,1,0,0.000000
284,JP,ODXST,2026-03-19,1,0,0.000000
448,JP,JPNX,2025-07-01,1,0,0.000000
95,JP,TOCOM,2026-06-05,1,0,0.000000
560,JP,ODXST,2026-07-02,1,0,0.000000
559,JP,ODX,2025-08-21,1,0,0.000000


## 3. Spoofing / layering

The detector flags a participant's cancel-ratio *z-score against their own trailing baseline* —
by design it catches a change in behavior, not a uniformly bad actor (see TRACKER.md Phase 5 for
why the first injected test case never triggered, and how it was redesigned). `P999` below is the
corrected test case: 10 baseline days, then a sustained spike from day 11 on.


In [6]:
spoof_df = q("""
SELECT EVENT_DATE, SUBMITTED_VOLUME, CANCELLED_UNFILLED_VOLUME, CANCEL_RATIO,
       BASELINE_PERIODS, CANCEL_RATIO_ZSCORE, IS_FLAGGED
FROM SPOOFING_LAYERING_SIGNALS
WHERE PARTICIPANT_ID = 'P999'
ORDER BY EVENT_DATE
""")
spoof_df


,EVENT_DATE,SUBMITTED_VOLUME,CANCELLED_UNFILLED_VOLUME,CANCEL_RATIO,BASELINE_PERIODS,CANCEL_RATIO_ZSCORE,IS_FLAGGED
0,2025-06-01,8000,4000,0.500000,0,NaN,False
1,2025-06-04,8000,4000,0.500000,1,NaN,False
2,2025-06-07,8000,4000,0.500000,2,NaN,False
3,2025-06-10,8000,4000,0.500000,3,NaN,False
4,2025-06-13,8000,4000,0.500000,4,NaN,False
5,2025-06-16,8000,4000,0.500000,5,NaN,False
6,2025-06-19,8000,4000,0.500000,6,NaN,False
7,2025-06-22,8000,4000,0.500000,7,NaN,False
8,2025-06-25,8000,4000,0.500000,8,NaN,False
9,2025-06-28,8000,4000,0.500000,9,NaN,False


In [7]:
flagged = spoof_df[spoof_df["IS_FLAGGED"]]
print(f"{len(flagged)} flagged day(s) for P999 -- first flag on "
      f"{flagged.iloc[0]['EVENT_DATE'] if len(flagged) else 'none'}, "
      "the first spike day once MIN_BASELINE_PERIODS=3 is satisfied.")


1 flagged day(s) for P999 -- first flag on 2025-07-04, the first spike day once MIN_BASELINE_PERIODS=3 is satisfied.


## 4. Position / exposure limits

`POSITION_LIMIT_BREACHES` reads its threshold entirely from `DETECTOR_CALIBRATION.PARAMS`
(`limit_quantity`), scoped by `JURISDICTION_ID` only — a concentration limit is a cross-venue
total, structurally consistent with `POSITIONS` itself carrying no `VENUE_ID`.


In [8]:
pos_df = q("SELECT * FROM POSITION_LIMIT_BREACHES ORDER BY PCT_OF_LIMIT DESC")
print(f"{len(pos_df)} position rows checked, {pos_df['IS_BREACH'].sum()} breach(es) "
      f"of a {pos_df['LIMIT_QUANTITY'].iloc[0]:,.0f}-unit limit.")
pos_df.head(10)


272 position rows checked, 1 breach(es) of a 50,000-unit limit.


,PARTICIPANT_ID,INSTRUMENT_ID,JURISDICTION_ID,AS_OF_DATE,NET_QUANTITY,MARKET_VALUE,CURRENCY,LIMIT_QUANTITY,PCT_OF_LIMIT,IS_BREACH
0,P011,I05,JP,2026-07-31,50917,73783316,JPY,50000,1.018340,True
1,P007,I07,JP,2026-07-19,48273,234781046,JPY,50000,0.965460,False
2,P000,I01,JP,2026-08-03,43611,145564796,JPY,50000,0.872220,False
3,P030,I07,JP,2026-05-24,42975,97971826,JPY,50000,0.859500,False
4,P005,I05,JP,2026-07-29,42665,26744982,JPY,50000,0.853300,False
5,P029,I02,JP,2026-08-01,41774,200642193,JPY,50000,0.835480,False
6,P019,I04,JP,2026-08-23,40553,90656232,JPY,50000,0.811060,False
7,P026,I00,JP,2026-08-06,40157,78223828,JPY,50000,0.803140,False
8,P031,I07,JP,2026-05-16,39816,69642166,JPY,50000,0.796320,False
9,P011,I07,JP,2026-08-06,39664,111914752,JPY,50000,0.793280,False


## 5. Reporting timeliness + template coverage

Two different claims, kept structurally distinct (Fix #28/#29): whether *this* report was
complete against the fields the schema can currently source (`REPORTING_TIMELINESS_SIGNALS`), and
whether the *template itself* has every required field mapped at all
(`REPORT_TEMPLATE_COVERAGE`). `Trading_Capacity` below is a deliberate `gap` field — required by
the regulator's template, no current source in `VIGIL.CORE` — surfaced, not fabricated or hidden.


In [9]:
timeliness_df = q("""
SELECT * FROM REPORTING_TIMELINESS_SIGNALS
WHERE IS_OVERDUE_UNSUBMITTED OR IS_LATE_SUBMISSION OR IS_INCOMPLETE OR IS_MISMATCHED
""")
print(f"{len(timeliness_df)} flagged reporting-timeliness row(s).")
timeliness_df.head(10)


145 flagged reporting-timeliness row(s).


,REPORT_ID,JURISDICTION_ID,VENUE_ID,REPORT_TYPE,REPORT_SCOPE,TRADE_ID,REPORT_STATUS,SUBMITTED_AT,DEADLINE,FIELDS_COMPLETE,MATCH_STATUS,DEFERRED_PUBLICATION_UNTIL,IS_OVERDUE_UNSUBMITTED,IS_LATE_SUBMISSION,IS_INCOMPLETE,IS_MISMATCHED
0,R0000780,JP,JPNX,transaction_report,trade,T0000781,new,2025-09-21 22:21:31,2025-09-22 20:21:31,False,full_match,None,False,False,True,False
1,R0000605,JP,CBOJBIDS,transaction_report,trade,T0000606,new,2025-08-03 23:43:19,2025-08-03 17:43:19,True,full_match,None,False,True,False,False
2,R0000407,JP,CBOJ,transaction_report,trade,T0000408,new,2025-07-16 10:21:11,2025-07-16 04:21:11,True,full_match,None,False,True,False,False
3,R0000099,JP,ODX,transaction_report,trade,T0000100,new,2026-06-06 21:50:38,2026-06-06 15:50:38,True,full_match,None,False,True,False,False
4,R0000374,JP,CBOJ,transaction_report,trade,T0000375,new,2025-06-11 02:43:48,2025-06-10 20:43:48,True,full_match,None,False,True,False,False
5,R0000814,JP,CBOJBIDS,transaction_report,trade,T0000815,new,2025-06-10 11:07:21,2025-06-10 05:07:21,True,full_match,None,False,True,False,False
6,R0000143,JP,JPNX,transaction_report,trade,T0000144,new,2025-07-16 14:28:38,2025-07-16 08:28:38,False,full_match,None,False,True,True,False
7,R0000884,JP,ODXST,transaction_report,trade,T0000885,new,2025-08-17 15:35:23,2025-08-18 13:35:23,False,full_match,None,False,False,True,False
8,R0000637,JP,ODX,transaction_report,trade,T0000638,new,2025-06-21 07:00:29,2025-06-22 05:00:29,False,full_match,None,False,False,True,False
9,R0000585,JP,ODX,transaction_report,trade,T0000586,new,2026-07-03 01:28:10,2026-07-03 23:28:10,False,full_match,None,False,False,True,False


In [10]:
coverage_tpl_df = q("SELECT * FROM REPORT_TEMPLATE_COVERAGE")
coverage_tpl_df


,JURISDICTION_ID,REPORT_TYPE,REQUIRED_FIELD_COUNT,MAPPED_REQUIRED_FIELD_COUNT,PCT_REQUIRED_FIELDS_MAPPED,GAP_FIELD_NAMES
0,JP,transaction_report,4,3,0.750000,"[\n ""Trading_Capacity""\n]"


## 6. Report assurance — one real report through `assure_report`

Pulls one real `TRANSACTION_REPORTS` row and the current `REPORT_TEMPLATES` for its
`(JURISDICTION_ID, REPORT_TYPE)`, builds the canonical source row from the underlying `TRADES`
row, and runs it through the same `assure_report.assure()` used by the `assure-report` skill
(Phase 6) — no query logic duplicated here, just wiring live rows into an already-tested function.


In [11]:
report_row = q("""
SELECT tr.*, t.PRICE, t.VOLUME, t.INSTRUMENT_ID
FROM TRANSACTION_REPORTS_CURRENT tr
JOIN TRADES t ON t.TRADE_ID = tr.TRADE_ID AND t.VENUE_ID = tr.VENUE_ID
WHERE tr.REPORT_TYPE = 'transaction_report'
LIMIT 1
""")
report_row.T


,0
REPORT_ID,R0000671
JURISDICTION_ID,JP
VENUE_ID,JPNX
REPORT_TYPE,transaction_report
REPORT_SCOPE,trade
TRADE_ID,T0000672
PERIOD_START,None
PERIOD_END,None
REPORT_STATUS,new
SUBMITTED_AT,2026-03-07 07:35:17


In [12]:
tpl_rows = q("""
SELECT FIELD_NAME, SOURCE_MAPPING, FIELD_FORMAT, IS_REQUIRED, STATUS
FROM REPORT_TEMPLATES_CURRENT
WHERE JURISDICTION_ID = 'JP' AND REPORT_TYPE = 'transaction_report'
""")
templates = [
    TemplateField(r.FIELD_NAME, r.SOURCE_MAPPING, r.FIELD_FORMAT, bool(r.IS_REQUIRED), r.STATUS)
    for r in tpl_rows.itertuples()
]

canonical_row = {
    "TRADES.PRICE": report_row.iloc[0]["PRICE"],
    "TRADES.VOLUME": report_row.iloc[0]["VOLUME"],
    "TRADES.INSTRUMENT_ID": report_row.iloc[0]["INSTRUMENT_ID"],
}
pct_mapped = coverage_tpl_df.loc[
    coverage_tpl_df["REPORT_TYPE"] == "transaction_report", "PCT_REQUIRED_FIELDS_MAPPED"
].iloc[0]

verdict = assure(templates, canonical_row, float(pct_mapped))
verdict


AssuranceVerdict(ready_to_submit=True, fields_complete=True, unresolved_required_fields=[], gap_fields=['Trading_Capacity'], reasons=["1 required field(s) have no data source yet (gap, not blocking): ['Trading_Capacity']", 'Template itself is only 75% mapped -- some required fields have no source at the template level, independent of this specific report.'])

## 7. Best execution

`EXECUTION_SLIPPAGE`/`ARRIVAL_SLIPPAGE` both depend on `TRADE_REFERENCE_PRICES`, which has its own
external adaptor (architecture.md) — the Phase 5 synthetic generator deliberately doesn't populate
it (out of its scope). Shown here for completeness, honestly reporting zero rows rather than
fabricating reference prices the generator never produced.


In [13]:
exec_slip = q("SELECT COUNT(*) AS N FROM EXECUTION_SLIPPAGE")
arrival_slip = q("SELECT COUNT(*) AS N FROM ARRIVAL_SLIPPAGE")
print(f"EXECUTION_SLIPPAGE: {exec_slip.iloc[0]['N']} rows "
      f"(TRADE_REFERENCE_PRICES not populated by the synthetic generator -- expected 0, not a bug)")
print(f"ARRIVAL_SLIPPAGE: {arrival_slip.iloc[0]['N']} rows (same reason)")


EXECUTION_SLIPPAGE: 0 rows (TRADE_REFERENCE_PRICES not populated by the synthetic generator -- expected 0, not a bug)
ARRIVAL_SLIPPAGE: 0 rows (same reason)


## 8. RBAC — a live positive/negative check

The full 27-check verification lives in `scripts/verify_rbac.py` (run and logged in Phase 2/7).
Here's one pair of checks inline, illustrating the governance gate structurally: `ANALYST_READ`
can read the wash-trading *view*, but not the base `OBLIGATION_MAP` table (the approval gate is a
grant, not a convention — `APPROVED_OBLIGATIONS` is the only obligation-lookup surface this role
has at all).


In [14]:
cur.execute("USE ROLE ANALYST_READ")
cur.execute("USE SECONDARY ROLES NONE")

cur.execute("SELECT COUNT(*) FROM WASH_TRADING_CANDIDATES")
print(f"ANALYST_READ -> SELECT WASH_TRADING_CANDIDATES: OK, {cur.fetchone()[0]} rows")

try:
    cur.execute("SELECT COUNT(*) FROM OBLIGATION_MAP")
    print("UNEXPECTED: ANALYST_READ could read base OBLIGATION_MAP -- RBAC gap")
except Exception as e:
    print(f"ANALYST_READ -> SELECT base OBLIGATION_MAP: correctly denied ({type(e).__name__})")

cur.execute("USE ROLE ACCOUNTADMIN")
cur.execute("USE SECONDARY ROLES NONE")
print('Role restored to ACCOUNTADMIN.')


ANALYST_READ -> SELECT WASH_TRADING_CANDIDATES: OK, 29 rows
ANALYST_READ -> SELECT base OBLIGATION_MAP: correctly denied (ProgrammingError)


Role restored to ACCOUNTADMIN.


## 9. Semantic View — the Cortex Analyst text-to-SQL surface

`SV_TRADE_SURVEILLANCE` is one of the two Semantic Views wired into `VIGIL_SURVEILLANCE_AGENT`
(Phase 6, `cortex_project/vigil_agent.sql`). Queried here directly via `SEMANTIC_VIEW()` — the
same mechanism Cortex Analyst uses under a natural-language question.


In [15]:
sv_df = q("""
SELECT * FROM SEMANTIC_VIEW(
    SV_TRADE_SURVEILLANCE
    DIMENSIONS VEN.VENUE_TYPE
    METRICS TRD.TRADE_COUNT, TRD.TOTAL_VOLUME, TRD.AVG_PRICE
)
ORDER BY TRADE_COUNT DESC
""")
sv_df


,VENUE_TYPE,TRADE_COUNT,TOTAL_VOLUME,AVG_PRICE
0,pts,499,2578659,2535.995992
1,exchange,302,1424084,2489.738411
2,block_trading,100,503772,2611.480000


## Summary

This walkthrough exercised, against real live data:
- the full core schema (Phase 1) and its append-only/milestoning discipline,
- RBAC's structural governance gate (Phase 2),
- all 4 detector families (Phase 3) — including the two real findings this session's behavioral
  validation caught (wash-trading NULL-exemption bug, spoofing test-case redesign — both in
  `TRACKER.md`),
- a Semantic View (Phase 4),
- the synthetic Japan dataset (Phase 5),
- two of the four product skills directly (`surveillance-query`, `assure-report`; the other two —
  `rule-interpret`, `narrative-draft` — are unit-tested in `tests/` but need a real rule corpus /
  audit-log lineage respectively to demo meaningfully, both explicitly flagged as backlog in
  `plan.md`).

Not covered here: the Cortex Agent's actual chat/orchestration behavior (`VIGIL_SURVEILLANCE_AGENT`
needs the Cortex Agents REST/chat interface, not plain SQL) and the Streamlit-in-Snowflake
dashboard (`ui/streamlit_app.py`, deployed as `VIGIL.CORE.VIGIL_DASHBOARD` — open it in Snowsight
for the interactive version of sections 2–7 above).


In [16]:
conn.close()
